In [ ]:
from typing import Any, Callable, Dict, Generator, List, Optional, Tuple, Type, Union

# 1. Recursion Patterns
**Teaching points during demo:**
- Visualize the call stack
- Show exponential growth of calls
- Demonstrate memoization benefits
- Discuss when to use recursion vs iteration

In [ ]:
# 2. Tree recursion (fibonacci)
def fib(n: int) -> int:
    if n <= 1:
        return n

    return fib(n-1) + fib(n-2)

# fib(5)
# fib(4)
# fib(3)
# fib(3)
# fib(2)

In [ ]:
# 3. Show inefficiency
import time

def time_function(func: Callable[[Any], Any], n: Any) -> Any:
    start = time.time()
    result = func(n)
    end = time.time()
    print(f"Time taken: {end - start:.4f} seconds")
    return result

def factorial(n: int) -> int:
    # Show call stack visualization here
    if n <= 1:  # Base case
        return 1

    return n * factorial(n - 1)  # Recursive case

# Compare performance
n = 3500
# Deep recursion deliberately exceeds Python's limit; catch it so Run All continues.
try:
    x = time_function(factorial, n)
except RecursionError:
    print('Recursion limit reached: use iteration for very deep recursion.')
    x = time_function(factorial, 100)

In [ ]:
time_function(fib, 35)

In [ ]:
# 4. Improved version with memoization
from functools import lru_cache

@lru_cache(maxsize=None)
def fib_cached(n):
    if n <= 1:
        return n
    return fib_cached(n-1) + fib_cached(n-2)

time_function(fib_cached, 35)

In [ ]:
cache = {0:0, 1:1} # key is the n, and value is the fib(n)
def fib_cache(n):
    if n not in cache:
        print(f'cache not found, evaluate fib({n})')
        cache[n] = fib_cache(n-1) + fib_cache(n-2)
        # del cash[n-2] # if you do this, you keep len(cache) < 3,which achieve O(1) space
    return cache[n]

In [ ]:
# Problem: Generate all possible combinations of parentheses
# Common interview question that's naturally recursive

def generate_parentheses_recursive(n: int) -> list[str]:
    def backtrack(open_n: int, closed_n: int, curr: str,
                 result: list[str]) -> None:
        # Base case: used all parentheses
        if len(curr) == 2 * n:
            result.append(curr)
            return

        # Add open parenthesis if we have any left
        if open_n < n:
            backtrack(open_n + 1, closed_n, curr + "(", result)

        # Add closing parenthesis if it's valid
        if closed_n < open_n:
            backtrack(open_n, closed_n + 1, curr + ")", result)

    result = []
    backtrack(0, 0, "", result)
    return result

In [ ]:
def generate_parentheses_iterative(n: int) -> list[str]:
    # Much more complex iterative solution
    stack = [(0, 0, "")]  # (open_n, closed_n, current_string)
    result = []

    while stack:
        open_n, closed_n, curr = stack.pop()

        if len(curr) == 2 * n:
            result.append(curr)
            continue

        # Add moves in reverse order (for same sequence as recursive)
        if closed_n < open_n:
            stack.append((open_n, closed_n + 1, curr + ")"))
        if open_n < n:
            stack.append((open_n + 1, closed_n, curr + "("))

    return result

In [ ]:
# Example usage:
n = 3
print("Recursive solution:", generate_parentheses_recursive(n))
print("Iterative solution:", generate_parentheses_iterative(n))

# Output for n=3:
# ["((()))", "(()())", "(())()", "()(())", "()()()"]

# 2. Context Managers

In [ ]:
import time

class Timer:
    def __init__(self, description: str) -> None:
        self.description: str = description
        self.start: Optional[float] = None
        self.end: Optional[float] = None

    def __enter__(self) -> "Timer":
        self.start = time.time()
        print(f"Starting {self.description}")
        return self

    def __exit__(
        self,
        exc_type: Optional[Type[BaseException]],
        exc_val: Optional[BaseException],
        exc_tb: Optional[object]
    ) -> None:
        self.end = time.time()
        print(f"{self.description} took {self.end - self.start:.2f} seconds")

In [ ]:
from contextlib import contextmanager
from typing import Generator

@contextmanager
def temporary_file(content: str) -> Generator[str, None, None]:
    # Setup
    filename: str = "temp.txt"
    with open(filename, 'w') as f:
        f.write(content)
    try:
        yield filename
    finally:
        # Cleanup
        import os
        os.remove(filename)

# Real-world example
with temporary_file("Hello, World!") as fname:
    with open(fname) as f:
        content: str = f.read()
        print(content)
# File is automatically cleaned up

#3. Async Programming

In [ ]:
import asyncio
import aiohttp
import time

# 1. Simple async function
async def say_hello(delay):
    await asyncio.sleep(delay)
    return f"Hello after {delay} seconds"

In [ ]:
x = say_hello(3)

In [ ]:
await x

In [ ]:
# 2. Running multiple coroutines
async def main():
    results = await asyncio.gather(
        say_hello(1),
        say_hello(2),
        say_hello(3)
    )
    print(results)

In [ ]:
await main()

In [ ]:
# 3. Real-world example: Async web requests
async def fetch_url(session, url):
    async with session.get(url, timeout=aiohttp.ClientTimeout(total=15)) as response:
        return await response.text()

async def fetch_multiple_urls(urls):
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_url(session, url) for url in urls]
        results = await asyncio.gather(*tasks)
        for url, result in zip(urls, results):
            print(f"Result from {url}: {result}")
        return results

# Compare with synchronous version
def sync_fetch_urls(urls):
    import requests
    return [requests.get(url, timeout=15).text for url in urls]

In [ ]:
# Function to time and compare sync vs async fetches
async def compare_fetch_methods(urls):
    # Time synchronous fetches
    print("Starting synchronous fetch...")
    start_sync = time.time()
    sync_results = sync_fetch_urls(urls)
    end_sync = time.time()
    sync_duration = end_sync - start_sync
    print(f"Synchronous fetch took {sync_duration:.2f} seconds")

    # Time asynchronous fetches
    print("Starting asynchronous fetch...")
    start_async = time.time()
    async_results = await fetch_multiple_urls(urls)
    end_async = time.time()
    async_duration = end_async - start_async
    print(f"Asynchronous fetch took {async_duration:.2f} seconds")

    # Print results comparison
    print("\nComparison:")
    print(f"Synchronous duration: {sync_duration:.2f} seconds")
    print(f"Asynchronous duration: {async_duration:.2f} seconds")

# List of example URLs
urls = [
    "https://jsonplaceholder.typicode.com/posts/1",
    "https://jsonplaceholder.typicode.com/posts/2",
    "https://jsonplaceholder.typicode.com/posts/3",
    "https://jsonplaceholder.typicode.com/posts/4",
    "https://jsonplaceholder.typicode.com/posts/5",
]

# Compare fetch methods
await compare_fetch_methods(urls)

# 4. Algorithm Complexity

Linear Search - "Finding by Checking Each Item"
Think of linear search like looking for a book in an unsorted stack. You start at the top and check each book, one by one, until you find what you're looking for. If the book isn't there, you'll need to check every single book to be sure. This is simple but gets slower as the stack gets bigger - if you have twice as many books, it takes twice as long to search.

In [ ]:
# 1. O(n) - Linear search
def linear_search(arr, target=999999):
    for i, num in enumerate(arr):
        if num == target:
            return i
    return -1

Binary Search - "Finding by Splitting in Half"
Binary search is like finding a word in a dictionary. Since the dictionary is already alphabetically sorted, you can open it to the middle and see if your word should be in the first or second half. Then you only look in that half, again splitting it in the middle. Each time, you eliminate half of the remaining pages! This is much faster than checking every page, but only works if the items are sorted first.

In [ ]:
# 2. O(log n) - Binary search
def binary_search(arr, target=999999):
    left, right = 0, len(arr) - 1
    while left <= right:
        mid = (left + right) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1

Merge Sort - "Sorting by Dividing and Combining"
Merge sort is like having a deck of cards and following these steps:
1. Split the deck in half
2. Split each half in half again (keep splitting until you have single cards)
3. Pick up two piles and merge them by taking the smaller card each time
4. Keep merging piles until you have one sorted pile

In [ ]:
# 3. O(n log n) - Merge Sort
def merge_sort(arr):
    if len(arr) <= 1:
        return arr

    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])

    return merge(left, right)

def merge(left, right):
    result = []
    i = j = 0

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1

    result.extend(left[i:])
    result.extend(right[j:])
    return result

Bubble Sort - "Sorting by Comparing Neighbors"
Bubble sort works like arranging cards in your hand. You look at two cards next to each other - if they're in the wrong order, you swap them. Then you move to the next pair. Each time you go through all the cards, at least one card "bubbles up" to its correct position. You keep doing this until no more swaps are needed. It's simple to understand but gets very slow with many items.

In [ ]:
# 4. O(n²) - Bubble sort
def bubble_sort(arr):
    n = len(arr)
    for i in range(n):
        for j in range(0, n-i-1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]

In [ ]:
import time
import matplotlib.pyplot as plt

def measure_time(func, sizes):
    times = []
    for size in sizes:
        data = list(range(size))
        start = time.time()
        func(data)
        end = time.time()
        times.append(end - start)
    return times



# Plot time complexity
sizes = [100, 500, 1000, 5000, 10000]
plt.figure(figsize=(10, 6))
plt.plot(sizes, measure_time(linear_search, sizes), label='Linear Search')
plt.plot(sizes, measure_time(binary_search, sizes), label='Binary Search')
plt.plot(sizes, measure_time(merge_sort, sizes), label='Merge Sort')
plt.plot(sizes, measure_time(bubble_sort, sizes), label='Bubble Sort')
plt.xlabel('Input Size')
plt.ylabel('Time (seconds)')
plt.legend()
plt.show()

In [ ]:
# Plot time complexity
sizes = [100, 500, 1000, 5000, 10000]
plt.figure(figsize=(10, 6))
plt.plot(sizes, measure_time(linear_search, sizes), label='Linear Search')
# plt.plot(sizes, measure_time(binary_search, sizes), label='Binary Search')
# plt.plot(sizes, measure_time(merge_sort, sizes), label='Merge Sort')
# plt.plot(sizes, measure_time(bubble_sort, sizes), label='Bubble Sort')
plt.xlabel('Input Size')
plt.ylabel('Time (seconds)')
plt.legend()
plt.show()

# 5. Data Structures Deep Dive

Binary Search Tree Explanation:
A Binary Search Tree (BST) is like a family tree for numbers, but with strict rules! Each "node" (think of it as a container) holds one value and can have up to two children - a left child and a right child. The cool rule is: anything smaller than a node goes to its left, anything larger goes to its right.

Our BST class lets us:
* Create a new empty tree (using `__init__`)
* Add new values (using insert) following our left/right rule
* Print all values in sorted order (using inorder)

The magic of BSTs is that they keep data organized - if you want to find a number, you can quickly eliminate half the tree at each step by just asking "is my number bigger or smaller than this node?"

In [ ]:
# 1. Implement Binary Search Tree
class TreeNode:
    def __init__(self, val=0):
        self.val = val
        self.left = None
        self.right = None

class BST:
    def __init__(self):
        self.root = None

    def insert(self, val):
        if not self.root:
            self.root = TreeNode(val)
            return

        def _insert(node, val):
            if val < node.val:
                if node.left is None:
                    node.left = TreeNode(val)
                else:
                    _insert(node.left, val)
            else:
                if node.right is None:
                    node.right = TreeNode(val)
                else:
                    _insert(node.right, val)

        _insert(self.root, val)

In [ ]:
# Example usage
def create_sample_bst():
    r"""
    Creates this tree:
         4
       /   \
      2     6
     / \   / \
    1   3 5   7
    """
    bst = BST()
    values = [4, 2, 6, 1, 3, 5, 7]
    for val in values:
        bst.insert(val)
    return bst

# Create and populate our BST
bst = create_sample_bst()

Preorder:
* Visit current node first
* Visit left child
* Visit right child

Use case: Great for creating a copy of the tree or serializing it!

Visual Analogy: Meet the parent before meeting their children

In [ ]:
# Let's also verify the tree structure is correct
def print_tree_structure(root, level=0, prefix="Root: "):
    if root is not None:
        print(" " * level + prefix + str(root.val))
        if root.left or root.right:
            print_tree_structure(root.left, level + 2, "L--- ")
            print_tree_structure(root.right, level + 2, "R--- ")

print("\nTree structure:")
print_tree_structure(bst.root)

In [ ]:
def preorder_traversal(root):
    result = []

    def _preorder(node):
        if node:
            result.append(node.val)      # Visit node
            _preorder(node.left)         # Go left
            _preorder(node.right)        # Go right

    _preorder(root)
    return result

print(f"Preorder:  {preorder_traversal(bst.root)}")   # Expected: [4, 2, 1, 3, 6, 5, 7]

Inorder:
* Visit left child
* Visit current node
* Visit right child

Result: Gets values in sorted order for a BST!

Visual Analogy: Visit youngest to oldest in each family branch

In [ ]:
def inorder_traversal(root):
    result = []

    def _inorder(node):
        if node:
            _inorder(node.left)          # Go left
            result.append(node.val)      # Visit node
            _inorder(node.right)         # Go right

    _inorder(root)
    return result

print(f"Inorder:   {inorder_traversal(bst.root)}")    # Expected: [1, 2, 3, 4, 5, 6, 7]

Postorder:
* Visit left child
* Visit right child
* Visit current node last

Use case: Perfect for deleting a tree or calculating folder sizes (need to process children before parent)!

Visual Analogy:  Meet all the children before meeting their parent

In [ ]:
def postorder_traversal(root):
    result = []

    def _postorder(node):
        if node:
            _postorder(node.left)        # Go left
            _postorder(node.right)       # Go right
            result.append(node.val)      # Visit node

    _postorder(root)
    return result

print(f"Postorder: {postorder_traversal(bst.root)}")  # Expected: [1, 3, 2, 5, 7, 6, 4]

Is Balanced Tree Explanation:
This function checks if a tree is "balanced" - meaning the left and right sides aren't too lopsided. It's like checking if a mobile hanging above a baby's crib is balanced!

Key points:

* A tree is balanced if the heights of left and right subtrees never differ by more than 1
* We use a clever trick: if we ever find an unbalanced part, we return -1 as a signal
* We calculate heights recursively (each node asks its children how tall they are)

In [ ]:
# 2. Common interview question
def is_balanced(root):
    def check_height(node):
        if not node:
            return 0

        left = check_height(node.left)
        if left == -1:
            return -1

        right = check_height(node.right)
        if right == -1:
            return -1

        if abs(left - right) > 1:
            return -1

        return max(left, right) + 1

    return check_height(root) != -1

In [ ]:
def create_balanced_tree():
    r"""
    Creates a balanced tree:
         4
       /   \
      2     6
     / \   / \
    1   3 5   7
    """
    bst = BST()
    values = [4, 2, 6, 1, 3, 5, 7]
    for val in values:
        bst.insert(val)
    return bst

def create_unbalanced_tree():
    r"""
    Creates an unbalanced tree:
         4
       /   \
      2     6
     /       \
    1         7
               \
                8
                 \
                  9
    """
    bst = BST()
    values = [4, 2, 6, 1, 7, 8, 9]
    for val in values:
        bst.insert(val)
    return bst

def print_tree_structure(root, level=0, prefix="Root: "):
    """Helper function to visualize the tree structure"""
    if root is not None:
        print(" " * level + prefix + str(root.val))
        if root.left or root.right:
            print_tree_structure(root.left, level + 2, "L--- ")
            print_tree_structure(root.right, level + 2, "R--- ")

# Test balanced tree
balanced_tree = create_balanced_tree()
print("Balanced Tree Structure:")
print_tree_structure(balanced_tree.root)
print(f"Is balanced? {is_balanced(balanced_tree.root)}\n")  # Should print True

# Test unbalanced tree
unbalanced_tree = create_unbalanced_tree()
print("Unbalanced Tree Structure:")
print_tree_structure(unbalanced_tree.root)
print(f"Is balanced? {is_balanced(unbalanced_tree.root)}\n")  # Should print False

# Let's create more test cases
def create_edge_case_trees():
    # Case 1: Single node tree (balanced)
    single_node = BST()
    single_node.insert(1)

    # Case 2: Empty tree (balanced)
    empty_tree = BST()

    # Case 3: Left-heavy unbalanced tree
    left_heavy = BST()
    left_heavy.root = TreeNode(5)
    left_heavy.root.left = TreeNode(4)
    left_heavy.root.left.left = TreeNode(3)

    return single_node, empty_tree, left_heavy

# Test edge cases
single_node, empty_tree, left_heavy = create_edge_case_trees()

print("Edge Cases:")
print("\n1. Single Node Tree:")
print_tree_structure(single_node.root)
print(f"Is balanced? {is_balanced(single_node.root)}")  # Should print True

print("\n2. Empty Tree:")
print_tree_structure(empty_tree.root)
print(f"Is balanced? {is_balanced(empty_tree.root)}")  # Should print True

print("\n3. Left-Heavy Tree:")
print_tree_structure(left_heavy.root)
print(f"Is balanced? {is_balanced(left_heavy.root)}")  # Should print False

# Demonstrate how adding one node can make an unbalanced tree balanced
def balance_left_heavy_tree(left_heavy):
    r"""
    Converts:        5             5
                   /     to      /   \
                  4            4     6
                 /
                3
    """
    left_heavy.root.right = TreeNode(6)
    return left_heavy

print("\nAfter balancing left-heavy tree:")
balanced_left_heavy = balance_left_heavy_tree(left_heavy)
print_tree_structure(balanced_left_heavy.root)
print(f"Is balanced? {is_balanced(balanced_left_heavy.root)}")  # Should print True

Hash Table Explanation:
Think of a hash table like a super-smart filing cabinet. Instead of searching through everything to find what you want, it uses a clever trick (the _hash function) to instantly know which drawer to look in!

Our HashTable:

* Creates "drawers" (we call them buckets) to store items
* Uses the _hash function to convert any key into a drawer number
* Can store (insert) and retrieve (get) items instantly
* Handles collisions (when two items want the same drawer) by making each drawer a list

In [ ]:
# 3. Hash Table implementation
class HashTable:
    def __init__(self, size=100):
        self.size = size
        self.table = [[] for _ in range(size)]

    def _hash(self, key):
        """
        Converts a key into a hash table index using a simple ASCII summation technique.

        Args:
            key: Any hashable type (will be converted to string)

        Returns:
            int: An index between 0 and (table_size - 1)

        Process:
            1. Convert key to string: str(key)
            - Handles any input type (numbers, strings, etc)
            - Example: 42 becomes "42", [1,2] becomes "[1, 2]"

            2. For each character, get ASCII value: ord(c)
            - ord('a') = 97, ord('b') = 98, etc
            - Example: "cat" becomes [99, 97, 116]

            3. Sum all ASCII values: sum(...)
            - Example: "cat" sums to 312 (99 + 97 + 116)

            4. Use modulo (%) with table size to get index
            - Ensures result is always between 0 and (size-1)
            - Example: if size=100, 312 % 100 = 12

        Examples:
            >>> ht = HashTable(size=10)
            >>> ht._hash("apple")  # (97 + 112 + 112 + 108 + 101) % 10 = 0
            0
            >>> ht._hash("cat")    # (99 + 97 + 116) % 10 = 2
            2
            >>> ht._hash(42)       # (52 + 50) % 10 = 2  # Note: collides with "cat"
            2

        Limitations:
            - Not cryptographically secure
            - Can have many collisions
            - Not uniform distribution for all inputs
            - Sensitive to character ordering (anagrams hash to same value)

        Note:
            This is a teaching-focused hash function that prioritizes simplicity
            and understandability over performance or security. Production systems
            typically use more sophisticated hashing algorithms like MurmurHash
            or xxHash.
        """
        return sum(ord(c) for c in str(key)) % self.size

    def insert(self, key, value):
        index = self._hash(key)
        for item in self.table[index]:
            if item[0] == key:
                item[1] = value
                return
        self.table[index].append([key, value])

    def get(self, key):
        index = self._hash(key)
        for item in self.table[index]:
            if item[0] == key:
                return item[1]
        raise KeyError(key)

In [ ]:
def demonstrate_hash_table():
    """
    Demonstrates various use cases and features of our HashTable implementation
    """
    print("Creating a HashTable with size 10...")
    ht = HashTable(size=10)

    # 1. Basic insertion and retrieval
    print("\n1. Basic Operations:")
    ht.insert("name", "John Doe")
    ht.insert("age", 30)
    ht.insert("city", "Berkeley")

    print(f"Retrieved name: {ht.get('name')}")
    print(f"Retrieved age: {ht.get('age')}")
    print(f"Retrieved city: {ht.get('city')}")

    # 2. Handling collisions
    print("\n2. Collision Handling:")
    # These will likely cause collisions in our simple hash function
    ht.insert("abc", 1)
    ht.insert("cba", 2)  # Different key that might hash to same value
    print(f"abc value: {ht.get('abc')}")
    print(f"cba value: {ht.get('cba')}")

    # 3. Updating existing keys
    print("\n3. Updating Values:")
    print(f"Original age: {ht.get('age')}")
    ht.insert("age", 31)  # Update age
    print(f"Updated age: {ht.get('age')}")

    # 4. Hash distribution visualization
    print("\n4. Hash Distribution:")
    def visualize_hash_distribution(hash_table):
        for i, bucket in enumerate(hash_table.table):
            print(f"Bucket {i}: {bucket}")

    visualize_hash_distribution(ht)

    # 5. Error handling
    print("\n5. Error Handling:")
    try:
        value = ht.get("nonexistent_key")
    except KeyError as e:
        print(f"Attempting to get nonexistent key resulted in: KeyError: {e}")

    # 6. Student record example
    print("\n6. Student Records Example:")
    student_db = HashTable(size=5)

    # Add student records
    students = [
        ("Alice", {"grade": "A", "student_id": "12345", "major": "CS"}),
        ("Bob", {"grade": "B", "student_id": "12346", "major": "Math"}),
        ("Charlie", {"grade": "A-", "student_id": "12347", "major": "Physics"})
    ]

    for name, record in students:
        student_db.insert(name, record)

    # Retrieve and update student records
    print(f"Alice's record: {student_db.get('Alice')}")

    # Update Bob's grade
    bob_record = student_db.get("Bob")
    bob_record["grade"] = "B+"
    student_db.insert("Bob", bob_record)

    print(f"Bob's updated record: {student_db.get('Bob')}")

    # 7. Performance demonstration
    print("\n7. Performance with Many Items:")
    big_table = HashTable(size=100)

    # Insert many items
    for i in range(50):
        key = f"key_{i}"
        big_table.insert(key, f"value_{i}")

    # Count items per bucket to check distribution
    bucket_sizes = [len(bucket) for bucket in big_table.table]
    print(f"Maximum items in a bucket: {max(bucket_sizes)}")
    print(f"Average items per bucket: {sum(bucket_sizes) / len(bucket_sizes):.2f}")

if __name__ == "__main__":
    demonstrate_hash_table()

# Additional example: Phone book implementation
def demonstrate_phone_book():
    print("\nPhone Book Example:")
    phone_book = HashTable(size=20)

    # Add contacts
    contacts = [
        ("John Smith", "510-123-4567"),
        ("Jane Doe", "510-234-5678"),
        ("Berkeley Pizza", "510-345-6789"),
        ("Campus Bookstore", "510-456-7890")
    ]

    for name, number in contacts:
        phone_book.insert(name, number)

    # Look up numbers
    try:
        print(f"John Smith's number: {phone_book.get('John Smith')}")
        print(f"Berkeley Pizza's number: {phone_book.get('Berkeley Pizza')}")
        print(f"Nonexistent contact's number: {phone_book.get('Nobody')}")
    except KeyError as e:
        print(f"Contact not found: {e}")

    # Update number
    phone_book.insert("John Smith", "510-999-9999")
    print(f"John Smith's new number: {phone_book.get('John Smith')}")

if __name__ == "__main__":
    demonstrate_hash_table()
    demonstrate_phone_book()

In [ ]:
# Red-Black Tree Implementation

from enum import Enum
from dataclasses import dataclass
from typing import Optional

class Color(Enum):
    RED = 0
    BLACK = 1

@dataclass
class Node:
    key: float
    color: Color = Color.RED
    left: Optional['Node'] = None
    right: Optional['Node'] = None
    parent: Optional['Node'] = None

class RedBlackTree:
    def __init__(self):
        self.NIL = Node(key=float('-inf'), color=Color.BLACK)
        self.root = self.NIL

    def left_rotate(self, x: Node) -> None:
        # Left rotation demonstration
        y = x.right  # y becomes new subtree root
        x.right = y.left  # y's left subtree becomes x's right subtree

        if y.left != self.NIL:
            y.left.parent = x

        y.parent = x.parent  # y takes x's place

        if x.parent == self.NIL:  # x was root
            self.root = y
        elif x == x.parent.left:  # x was left child
            x.parent.left = y
        else:  # x was right child
            x.parent.right = y

        y.left = x  # x becomes y's left child
        x.parent = y

    def insert_fixup(self, k: Node) -> None:
        # Example of color fixing after insert
        while k.parent.color == Color.RED:
            if k.parent == k.parent.parent.right:
                u = k.parent.parent.left
                if u.color == Color.RED:
                    # Case 1: Uncle is red
                    u.color = Color.BLACK
                    k.parent.color = Color.BLACK
                    k.parent.parent.color = Color.RED
                    k = k.parent.parent
                else:
                    if k == k.parent.left:
                        # Case 2: Uncle is black, k is left child
                        k = k.parent
                        self.right_rotate(k)
                    # Case 3: Uncle is black, k is right child
                    k.parent.color = Color.BLACK
                    k.parent.parent.color = Color.RED
                    self.left_rotate(k.parent.parent)
            else:
                # Mirror image cases
                pass  # Similar to above with left/right exchanged

        self.root.color = Color.BLACK  # Ensure root is black

# Example usage:
tree = RedBlackTree()

# Insert values (this would trigger rotations and recoloring)
prices = [100.50, 100.55, 100.45, 100.60]
for price in prices:
    # Full insert implementation would go here
    print(f"Inserting {price}")

In [ ]:
# Installed with the course environment: uv sync --locked
import sortedcontainers


In [ ]:
# Order Book Implementation

from dataclasses import dataclass
from typing import Dict, Optional
from sortedcontainers import SortedDict  # Red-black tree implementation
from collections import defaultdict

@dataclass
class Order:
    order_id: str
    price: float
    quantity: int
    is_buy: bool

class OrderList:
    def __init__(self):
        self.head = None
        self.tail = None
        self.orders: Dict[str, Order] = {}
        self.total_volume = 0

    def add_order(self, order: Order) -> None:
        self.orders[order.order_id] = order
        self.total_volume += order.quantity

    def remove_order(self, order_id: str) -> Optional[Order]:
        if order_id in self.orders:
            order = self.orders.pop(order_id)
            self.total_volume -= order.quantity
            return order
        return None

class OrderBook:
    def __init__(self):
        # Price level -> OrderList mapping
        self.buy_orders = SortedDict()   # Sorted high to low
        self.sell_orders = SortedDict()  # Sorted low to high

        # Quick order lookup
        self.orders: Dict[str, Order] = {}

        # Price level volumes
        self.buy_volume = defaultdict(int)
        self.sell_volume = defaultdict(int)

    def add_order(self, order: Order) -> None:
        # Choose correct price level structure
        book = self.buy_orders if order.is_buy else self.sell_orders
        volume_map = self.buy_volume if order.is_buy else self.sell_volume

        # Create price level if it doesn't exist
        if order.price not in book:
            book[order.price] = OrderList()

        # Add to price level and update volume
        book[order.price].add_order(order)
        volume_map[order.price] += order.quantity
        self.orders[order.order_id] = order

    def get_best_bid(self) -> Optional[float]:
        return self.buy_orders.peekitem(-1)[0] if self.buy_orders else None

    def get_best_ask(self) -> Optional[float]:
        return self.sell_orders.peekitem(0)[0] if self.sell_orders else None

    def get_volume_at_price(self, price: float, is_buy: bool) -> int:
        return (self.buy_volume[price] if is_buy
                else self.sell_volume[price])

# Example usage:
book = OrderBook()

# Add some orders
order1 = Order("1", 100.50, 100, True)   # Buy
order2 = Order("2", 100.60, 200, True)   # Buy
order3 = Order("3", 100.70, 150, False)  # Sell

book.add_order(order1)
book.add_order(order2)
book.add_order(order3)

print(f"Best bid: {book.get_best_bid()}")
print(f"Best ask: {book.get_best_ask()}")
print(f"Volume at 100.50: {book.get_volume_at_price(100.50, True)}")

# 6. Memory Management

In [ ]:
# Installed with the course environment: uv sync --locked
import memory_profiler


In [ ]:
import gc
import weakref
from memory_profiler import memory_usage
import time

Memory Leaks with Circular References

* What it does: Shows how objects can get stuck in memory when they reference each other in a circle.
* Real-world example: Imagine two friends on social media who follow each other. Even if both delete their accounts, their data might stay in memory because each account is still being "referenced" by the other.
* In the code, we create two Node objects that point to each other (through next and prev). Even when we try to delete them, they stay in memory because they're still referencing each other.

In [ ]:
print("=== Example 1: Memory Leaks Demo ===")

class Node:
    def __init__(self, value):
        self.value = value
        self.next = None
        self.prev = None  # Creates potential for circular reference

    def __repr__(self):
        return f"Node({self.value})"

# Let's create and examine a memory leak
node1 = Node(1)
node2 = Node(2)
node1.next = node2
node2.prev = node1

# Check initial state
print(f"Initial reference count for node1: {gc.get_referrers(node1).__len__()}")

# Try to delete references
del node1
del node2

# Force garbage collection and see what happens
collected = gc.collect()
print(f"Number of objects collected by gc: {collected}")

Weak References

* What it does: Demonstrates a solution to circular references using "weak" references that don't prevent garbage collection.
* Real-world example: Think of it like keeping a person's business card (weak reference) versus having them as an emergency contact (strong reference). If they move away, the business card doesn't keep them from leaving.
* The code shows how using weakref.ref() allows objects to be cleaned up properly even when they reference each other.

In [ ]:
print("\n=== Example 2: Weak References Demo ===")

class NodeWithWeakref:
    def __init__(self, value):
        self.value = value
        self.next = None
        self.prev = None  # Initialize as None first

    def __repr__(self):
        return f"NodeWithWeakref({self.value})"

# Create nodes with weak references
node1 = NodeWithWeakref(1)
node2 = NodeWithWeakref(2)

# Set up the references
node1.next = node2  # Strong reference from node1 to node2
node2.prev = weakref.ref(node1)  # Weak reference from node2 back to node1

# Check weak reference
print(f"Node1 through weak reference: {node2.prev()}")

# Delete reference to node1
del node1

# Check weak reference again - should be None
print(f"Node1 through weak reference after deletion: {node2.prev()}")

Memory Profiling

* What it does: Shows the difference between code that uses memory efficiently versus inefficiently.
* Real-world example: It's like the difference between writing down every number you see before calculating their average (inefficient) versus just keeping a running total and count (efficient).
* The code compares two ways to calculate an average: one that stores all numbers in a list (using more memory) and one that just keeps a running total (using less memory).

We sample process memory with `memory_usage`, which works directly in notebooks. Both functions average the same one million integers. Peak process memory includes the notebook kernel; it is not just the list size.

In [ ]:
print("\n=== Example 3a: Memory Profiling - Inefficient Version ===")

def inefficient_processing():
    """
    Demonstrates inefficient memory usage by storing all numbers in a list.
    """
    numbers = []
    for i in range(1_000_000):
        numbers.append(i)
    return sum(numbers) / len(numbers)

# Run inefficient version
start = time.time()
peak1, result1 = memory_usage((inefficient_processing,), interval=0.01, max_usage=True, retval=True)
print(f'Peak process memory: {peak1:.1f} MiB')
print(f"Inefficient result: {result1:.2f}")
print(f"Time taken: {time.time() - start:.2f}s")

In [ ]:
print("\n=== Example 3b: Memory Profiling - Efficient Version ===")

def efficient_processing():
    """
    Demonstrates efficient memory usage using a generator-like approach.
    Uses range properties instead of counting manually.
    """
    n = 1_000_000
    # Use arithmetic sequence sum formula: sum = n(a1 + an)/2
    # where n is count, a1 is first term (0), and an is last term (n-1)
    total = (n * (n - 1)) // 2
    return total / n


# Run efficient version
start = time.time()
peak2, result2 = memory_usage((efficient_processing,), interval=0.01, max_usage=True, retval=True)
print(f'Peak process memory: {peak2:.1f} MiB')
print(f"Efficient result: {result2:.2f}")
print(f"Time taken: {time.time() - start:.2f}s")

In [ ]:
print("\n=== Results Comparison ===")
print(f"Both methods give same result: {abs(result1 - result2) < 0.0001}")

# Optional: Check current garbage collector state
print("\nGarbage collector state:")
print(f"GC counts: {gc.get_count()}")
print(f"GC enabled: {gc.isenabled()}")

# 7. Concurrency Patterns

In [ ]:
import asyncio
import aiohttp
import requests
import time
import math
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from typing import List
from datetime import datetime

In [ ]:
# 1. IO-bound task using ThreadPoolExecutor
def download_url(url: str) -> tuple[str, float]:
    start_time = time.time()
    response = requests.get(url, timeout=15).text
    return response[:100], time.time() - start_time  # Return first 100 chars and time taken

def demonstrate_threading():
    print("\n=== Threading Example (IO-bound) ===")
    urls = ['https://example.com'] * 8

    # Sequential execution for comparison
    start = time.time()
    sequential_results = [download_url(url) for url in urls]
    print(f"Sequential execution took: {time.time() - start:.2f} seconds")

    # Parallel execution with ThreadPoolExecutor
    start = time.time()
    with ThreadPoolExecutor(max_workers=4) as executor:
        parallel_results = list(executor.map(download_url, urls))
    print(f"Threaded execution took: {time.time() - start:.2f} seconds")

demonstrate_threading()

In [ ]:
# 3. IO-bound task using asyncio + semaphore
async def fetch_url_async(session: aiohttp.ClientSession, url: str, semaphore: asyncio.Semaphore) -> tuple[str, float]:
    start_time = time.time()
    async with semaphore:  # Limit concurrent requests
        async with session.get(url, timeout=aiohttp.ClientTimeout(total=15)) as response:
            text = await response.text()
            return text[:100], time.time() - start_time

async def demonstrate_asyncio():
    print("\n=== Asyncio Example (IO-bound) ===")
    urls = ['https://example.com'] * 8
    semaphore = asyncio.Semaphore(4)  # Limit concurrent requests

    async with aiohttp.ClientSession() as session:
        # Create tasks for all URLs
        tasks = [
            fetch_url_async(session, url, semaphore)
            for url in urls
        ]

        # Execute all tasks concurrently
        start = time.time()
        results = await asyncio.gather(*tasks)
        print(f"Asyncio execution took: {time.time() - start:.2f} seconds")

        return results

await demonstrate_asyncio()

In [ ]:
# 2. CPU-bound task using ProcessPoolExecutor
# An importable worker works with Jupyter on macOS, Windows, and Linux.
from lecture6_workers import calculate_primes

def demonstrate_multiprocessing():
    print("\n=== Multiprocessing Example (CPU-bound) ===")
    numbers = [50000] * 16

    # Sequential execution for comparison
    start = time.time()
    sequential_results = [calculate_primes(n) for n in numbers]
    print(f"Sequential execution took: {time.time() - start:.2f} seconds")

    # Parallel execution with ProcessPoolExecutor
    start = time.time()
    with ProcessPoolExecutor(max_workers=4) as executor:
        parallel_results = list(executor.map(calculate_primes, numbers))
    print(f"Multiprocessing execution took: {time.time() - start:.2f} seconds")

demonstrate_multiprocessing()

In [ ]:
# 4. Utility function to run and compare all approaches
async def compare_approaches():
    # Run threading example
    demonstrate_threading()

    # Run asyncio example
    await demonstrate_asyncio()

    # Run multiprocessing example
    demonstrate_multiprocessing()

print(f"Starting concurrent programming demonstrations at {datetime.now()}")
print("Each example will show sequential vs concurrent execution times")
print("-" * 50)

await compare_approaches()

# 8. Error Handling

In [ ]:
# Custom exceptions for domain-specific errors
class TradingError(Exception):
    """Base class for trading-related errors"""
    pass

class InsufficientFundsError(TradingError):
    def __init__(self, balance, amount):
        self.balance = balance
        self.amount = amount
        super().__init__(
            f"Insufficient funds: {balance} < {amount}"
        )

# EAFP approach (Pythonic)
def execute_trade_eafp(symbol, amount):
    try:
        price = get_current_price(symbol)
        execute_order(symbol, amount, price)
    except KeyError:
        logger.error(f"Invalid symbol: {symbol}")
        raise TradingError(f"Unknown symbol: {symbol}")
    except InsufficientFundsError as e:
        logger.warning(f"Trade failed: {e}")
        raise
    except Exception as e:
        logger.exception("Unexpected trading error")
        raise TradingError(f"Trade failed: {str(e)}")
    finally:
        cleanup_trading_session()

# LBYL approach (less preferred)
def execute_trade_lbyl(symbol, amount):
    if symbol not in valid_symbols:
        raise TradingError(f"Unknown symbol: {symbol}")
    if get_balance() < amount:
        raise InsufficientFundsError(get_balance(), amount)

    price = get_current_price(symbol)
    execute_order(symbol, amount, price)

In [ ]:
class ValidationError(Exception):
    pass

class InsufficientFundsError(Exception):
    def __init__(self, balance, amount):
        self.balance = balance
        self.amount = amount
        super().__init__(
            f"Insufficient funds: balance={balance}, amount={amount}"
        )

In [ ]:
# 2. Context-based error handling
class BankAccount:
    def __init__(self, balance):
        if balance < 0:
            raise ValidationError("Initial balance cannot be negative")
        self.balance = balance

    def withdraw(self, amount):
        if amount > self.balance:
            raise InsufficientFundsError(self.balance, amount)
        self.balance -= amount

    def deposit(self, amount):
        if amount <= 0:
            raise ValidationError("Deposit amount must be positive")
        self.balance += amount

In [ ]:
# 3. Error handling patterns
def process_transaction(account, amount):
    try:
        account.withdraw(amount)
    except InsufficientFundsError as e:
        logger.error(f"Transaction failed: {e}")
        # Automatic retry with smaller amount
        try_amount = e.balance
        account.withdraw(try_amount)
    except ValidationError as e:
        logger.error(f"Validation failed: {e}")
        raise
    except Exception as e:
        logger.exception("Unexpected error")
        raise

# 9. Logging Setup

In [ ]:
import logging
import json
from datetime import datetime, timezone
from logging.handlers import RotatingFileHandler

class JSONFormatter(logging.Formatter):
    def format(self, record):
        log_obj = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "level": record.levelname,
            "message": record.getMessage(),
            "module": record.module,
            "function": record.funcName,
            "transaction_id": getattr(record, "transaction_id", None)
        }
        return json.dumps(log_obj)

def setup_logging():
    # Create logger
    logger = logging.getLogger("trading_system")
    logger.setLevel(logging.DEBUG)

    # JSON formatter for files
    json_formatter = JSONFormatter()

    # Console handler (INFO and above)
    console = logging.StreamHandler()
    console.setLevel(logging.INFO)
    console.setFormatter(logging.Formatter(
        '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
    ))

    # File handler with rotation (all levels)
    file_handler = RotatingFileHandler(
        "trading.log",
        maxBytes=10_000_000,  # 10MB
        backupCount=5
    )
    file_handler.setFormatter(json_formatter)

    # Error file handler (ERROR and above)
    error_handler = RotatingFileHandler(
        "error.log",
        maxBytes=10_000_000,
        backupCount=5
    )
    error_handler.setLevel(logging.ERROR)
    error_handler.setFormatter(json_formatter)

    # Add all handlers
    logger.addHandler(console)
    logger.addHandler(file_handler)
    logger.addHandler(error_handler)

    return logger

# Usage example
logger = setup_logging()
logger.info("Trade executed", extra={
    "transaction_id": "TX123",
    "symbol": "AAPL",
    "amount": 100
})